<a href="https://colab.research.google.com/github/Brafied/defect-detection-active-learning/blob/braden/visa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Download and extract the Visa dataset
!wget --show-progress https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar
!tar -xf VisA_20220922.tar

In [ ]:
# Clone the spot-diff repository
!git clone https://github.com/amazon-science/spot-diff.git

In [ ]:
# Split configurations (see table below)
cfgs = {
    "1cls": {
        "save_folder": "./VisA_1cls",
        "split_file": "./spot-diff/split_csv/1cls.csv",
    },
    "2cls_highshot": {
        "save_folder": "./VisA_2cls_highshot",
        "split_file": "./spot-diff/split_csv/2cls_highshot.csv",
    },
    "2cls_fewshot": {
        "save_folder": "./VisA_2cls_fewshot",
        "split_file": "./spot-diff/split_csv/2cls_fewshot.csv",
    },
}

| Split | Training set data | Testing set data |
|---|---|---|
| `1cls` | 90% of normal images | 10% of normal images + 100% of anomalous images |
| `2cls_highshot` | 60% of normal images + 60% of anomalous images | 40% of normal images + 40% of anomalous images |
| `2cls_fewshot` | 20% of normal images + 20% of anomalous images | 80% of normal images + 80% of anomalous images |

In [ ]:
# Select split configuration and process data
split_type = "2cls_highshot"
cfg = cfgs[split_type]
!python spot-diff/utils/prepare_data.py \
  --split-type {split_type} \
  --data-folder . \
  --save-folder {cfg["save_folder"]} \
  --split-file {cfg["split_file"]}

In [ ]:
# Remove tarball and unprocessed data
!rm -f VisA_20220922.tar
!rm -rf candle capsules cashew chewinggum fryum macaroni1 macaroni2 pcb1 pcb2 pcb3 pcb4 pipe_fryum

In [ ]:
import torch.nn as nn
from transformers import AutoModel

# The DefectClassifier consists of a pretrained encoder and a binary classification head
class DefectClassifier(nn.Module):
    def __init__(self, model_id):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        self.classification_head = nn.Linear(self.encoder.config.hidden_sizes[-1], 1)

    def forward(self, pixel_values):
        features = self.encoder(pixel_values)
        logits = self.classification_head(features.pooler_output).squeeze(-1)
        return logits

In [ ]:
from transformers import AutoImageProcessor
import torch

MODEL_ID = "facebook/dinov3-convnext-tiny-pretrain-lvd1689m"

# The processor converts images into the format the encoder is expecting
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = DefectClassifier(MODEL_ID).to(device)
# Disable training of the encoder weights
for parameter in model.encoder.parameters():
    parameter.requires_grad = False

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), 1e-5, weight_decay=1e-4)

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class VisaDatasetSplit(Dataset):
    # Parse the images in the specified split directory
    def __init__(self, split_directory):
        self.examples = []
        for path in (split_directory / "good").glob("*"):
            self.examples.append((path, 0))
        for path in (split_directory / "bad").glob("*"):
            self.examples.append((path, 1))

    def __len__(self):
        length = len(self.examples)
        return length

    # Apply the processor to the image at the specified index. Return the result and corresponding label.
    def __getitem__(self, index):
        path, label = self.examples[index]
        pixel_values = processor(Image.open(path))["pixel_values"][0]
        label = torch.tensor(label, dtype=torch.float32)
        return pixel_values, label

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader

item = "candle"
root_directory = Path(cfg["save_folder"]) / split_type / item

training_split = VisaDatasetSplit(root_directory / "train")
testing_split = VisaDatasetSplit(root_directory / "test")

training_loader = DataLoader(training_split, 16, True)
testing_loader = DataLoader(testing_split, 16, False)

In [ ]:
epoch_count = 10
for current_epoch in range(epoch_count):
    training_loss = 0.0
    model.train()
    for pixel_values, labels in training_loader:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(pixel_values)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()

        training_loss += loss.item() * labels.size(0)

    testing_correct = 0
    model.eval()
    with torch.no_grad():
        for pixel_values, labels in testing_loader:
            pixel_values = pixel_values.to(device)

            logits = model(pixel_values)

            testing_correct += ((torch.sigmoid(logits) >= 0.5).cpu() == labels.bool()).sum().item()

    print(f"Current epoch: {current_epoch + 1}/{epoch_count},  training loss: {training_loss / len(training_split):.4f}, testing accuracy: {testingg_correct / len(testing_split):.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    for pixel_values, labels in testing_loader:
        pixel_values = pixel_values.to(device)
        logits = model(pixel_values)
        print((torch.sigmoid(logits) >= 0.5).cpu())